<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 강화학습·에이전트 · 01. DQN

## Human-level Control through Deep Reinforcement Learning

- **원 논문:** [Human-level Control through Deep Reinforcement Learning](https://www.nature.com/articles/nature14236)
- **저자 / 발표:** Volodymyr Mnih et al. · Nature (2015)
- **난이도 / 예상 시간:** 중급 · 약 55분
- **선수 지식:** Bellman equation, replay, target network
- **로컬 학습 데이터:** `data/field_curriculum/gridworld.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 로컬 transition replay에서 one-hot DQN과 frozen target network의 TD loss를 학습한다.

**축소하거나 생략한 부분:** 논문은 raw Atari frames, convolutional Q-network, 대규모 replay를 사용한다. 축소판은 동일한 target/replay mechanics를 작은 MDP에 적용한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | Methods, ‘Deep reinforcement learning’, Eq. (1)–(2) | squared TD loss and frozen target | r+γmax Q_target로 online Q를 회귀한다. |
| Methods, experience replay paragraph | uniform replay minibatches | 준비된 transition rows에서 무작위 mini-batch를 뽑는다. |
| Extended Data, Algorithm 1 | DQN training loop | epsilon interaction 대신 offline replay를 사용하되 update와 target sync를 보존한다. |

## 핵심 재현

        Methods Eq. (2)의 target $y=r+\gamma(1-d)\max_{a'}Q_{target}(s',a')$를
        구현합니다. target tensor가 gradient graph와 분리되는지도 검사합니다.

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(411)
np.random.seed(411)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# Tiny scalar/table loops may be faster on CPU even when a GPU exists;
# AI_LAB_DEVICE=cpu remains the low-overhead override.
DATA_PATH = Path("data/field_curriculum/gridworld.json")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
config = json.loads(DATA_PATH.read_text(encoding="utf-8"))
required = {
    "seed",
    "size",
    "start_state",
    "terminal_state",
    "gamma",
    "transitions",
    "bandit_means",
    "offline_action_noise",
}
assert required.issubset(config), f"JSON 키 불일치: {sorted(config)}"
rows = config["transitions"]
states = torch.tensor([int(r["state"]) for r in rows], dtype=torch.long)
actions = torch.tensor([int(r["action"]) for r in rows], dtype=torch.long)
rewards = torch.tensor([float(r["reward"]) for r in rows], dtype=torch.float32)
next_states = torch.tensor([int(r["next_state"]) for r in rows], dtype=torch.long)
dones = torch.tensor([float(r["done"]) for r in rows], dtype=torch.float32)
n_states = int(max(states.max(), next_states.max()).item()) + 1
n_actions = int(actions.max().item()) + 1
gamma = float(config["gamma"])
start_state, terminal_state = int(config["start_state"]), int(config["terminal_state"])
transition_table = {
    (int(r["state"]), int(r["action"])): (
        int(r["next_state"]),
        float(r["reward"]),
        bool(r["done"]),
    )
    for r in rows
}


def encode(s):
    return F.one_hot(torch.as_tensor(s, dtype=torch.long), n_states).float()


def encode_device(s):
    return encode(s).to(DEVICE)


def env_step(state, action):
    return transition_table[(int(state), int(action))]


def value_iteration(iterations=300):
    q = torch.zeros(n_states, n_actions)
    for _ in range(iterations):
        old = q.clone()
        for (s, a), (ns, r, done) in transition_table.items():
            q[s, a] = r if done else r + gamma * old[ns].max()
    return q


q_star = value_iteration()
assert torch.isfinite(q_star).all() and len(rows) > 0
print(ACCELERATOR.summary())
print(
    "RL environment, replay metadata, and tree search stay on CPU; "
    "neural batches use DEVICE."
)
print(
    f"local MDP: states={n_states}, actions={n_actions}, "
    f"transitions={len(rows)}, gamma={gamma}"
)

## 포트폴리오 구현 설계: 수식 → 에이전트 책임 → 검증

논문의 핵심 update를 실행 가능한 에이전트의 `forward`, `loss`, `update`,
`evaluate`로 나눕니다. 아래 식의 선택·평가·gradient 경계를 메서드 본문에서
한 줄씩 추적하세요.

$$
y_t=r_t+\gamma(1-d_t)\max_{a'}Q_{\theta^-}(s_{t+1},a'),\qquad \mathcal L=\mathbb E[(y_t-Q_\theta(s_t,a_t))^2]
$$

- **기호와 역할:** $Q_\theta$는 online network, $Q_{\theta^-}$는 frozen target, $d_t$는 terminal mask입니다.
- **shape/state 계약:** `one-hot state [B, S] → Q [B, A]; action/reward/done [B]`
- **논문 위치:** `Methods, ‘Deep reinforcement learning’, Eq. (1)–(2)`의 **squared TD loss and frozen target**
- **코드 Task:** replay batch, detached Bellman target, Huber loss와 target sync를 구현합니다.
- **학습·평가 흐름:** uniform replay → online gather → target max → TD update → 주기적 sync/평가
- **원문 대비 한계:** 논문은 raw Atari frames, convolutional Q-network, 대규모 replay를 사용한다. 축소판은 동일한 target/replay mechanics를 작은 MDP에 적용한다.
- **구현 이유:** target, gradient, optimizer 책임을 Agent 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 Bellman target, policy objective, search score 같은 핵심 계산을
불투명한 trainer 뒤에 감추지 않습니다. 실습본에서는 공개 API를 유지하면서 TODO를
채우고, 각 메서드의 입력 shape와 gradient가 끊기는 위치를 설명하세요.

In [ ]:
class DqnAgent(nn.Module):
    """DQN의 핵심 학습·평가 경로. 입출력 계약: one-hot state [B, S] → Q [B, A];
    action/reward/done [B]."""

    def __init__(self, online_network, target_network, optimizer, discount):
        """network, optimizer, target 또는 table 상태를 명시적으로 저장한다."""
        super().__init__()
        self.online_network = online_network
        self.target_network = target_network
        self.optimizer = optimizer
        self.discount = discount

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {
            "state_count": n_states,
            "action_count": n_actions,
            "gamma": self.discount,
        }

    def forward(self, state):
        """policy/value/Q 순전파를 계산한다. shape 계약: one-hot state [B, S] → Q [B, A];
        action/reward/done [B]."""
        encoded_state = encode_device(state)
        return self.online_network(encoded_state)

    def loss(self, state, action, reward, next_state, done):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        predicted_q = self.forward(state).gather(1, action[:, None]).squeeze(1)
        with torch.no_grad():
            encoded_next_state = encode_device(next_state)
            next_q = self.target_network(encoded_next_state).max(dim=1).values
            target_q = reward + self.discount * (1.0 - done) * next_q
        return F.smooth_l1_loss(predicted_q, target_q)

    def update(self, state, action, reward, next_state, done):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        objective = self.loss(state, action, reward, next_state, done)
        self.optimizer.zero_grad()
        objective.backward()
        torch.nn.utils.clip_grad_norm_(self.online_network.parameters(), 10.0)
        self.optimizer.step()
        return float(objective.detach())

    def sync_target(self):
        """online network의 가중치를 target network로 복사한다."""
        self.target_network.load_state_dict(self.online_network.state_dict())

    def evaluate(self, reference_q):
        """no_grad 경로에서 정책 또는 value 품질 지표 dict를 반환한다."""
        with torch.no_grad():
            learned_q = self.forward(torch.arange(n_states))
            nonterminal = torch.arange(n_states, device=DEVICE) != terminal_state
            agreement = (
                (
                    learned_q.argmax(dim=1)[nonterminal]
                    == reference_q.to(DEVICE).argmax(dim=1)[nonterminal]
                )
                .float()
                .mean()
            )
        return {"greedy_agreement": float(agreement)}

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `Methods, ‘Deep reinforcement learning’, Eq. (1)–(2)` — **squared TD loss and frozen target**
- **구현 이유:** r+γmax Q_target로 online Q를 회귀한다.
- **읽을 코드:** 아래 `implementation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
def dqn_target(r, done, next_q, discount):
    return (r + discount * (1 - done) * next_q.max(1).values).detach()


probe = dqn_target(
    torch.tensor([1.0, 2.0]),
    torch.tensor([0.0, 1.0]),
    torch.tensor([[3.0, 4.0], [9.0, 8.0]], requires_grad=True),
    0.5,
)
assert torch.allclose(probe, torch.tensor([3.0, 2.0])) and not probe.requires_grad

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `Extended Data, Algorithm 1` — **DQN training loop**
- **구현 이유:** epsilon interaction 대신 offline replay를 사용하되 update와 target sync를 보존한다.
- **읽을 코드:** 아래 `training` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
def qnet():
    return nn.Sequential(
        nn.Linear(n_states, 32), nn.ReLU(), nn.Linear(32, n_actions)
    ).to(DEVICE)


online, target = qnet(), qnet()
target.load_state_dict(online.state_dict())
opt = torch.optim.Adam(online.parameters(), lr=0.01)
rng = torch.Generator().manual_seed(412)
history = []

portfolio_agent = DqnAgent(online, target, opt, gamma)
initial_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in online.parameters()]
)
for step in range(260):
    index = torch.randint(len(rows), (min(32, len(rows)),), generator=rng)
    state, action, reward, next_state, done = ACCELERATOR.move(
        states[index],
        actions[index],
        rewards[index],
        next_states[index],
        dones[index],
    )
    history.append(portfolio_agent.update(state, action, reward, next_state, done))
    if (step + 1) % 20 == 0:
        portfolio_agent.sync_target()
final_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in online.parameters()]
)
parameter_delta = float((final_parameters - initial_parameters).abs().sum())
assert parameter_delta > 0.0 and np.isfinite(history).all()
assert np.mean(history[-20:]) < np.mean(history[:20])

### 단계 4. 평가·시각화

- **논문의 어느 부분인가:** `Methods, experience replay paragraph` — **uniform replay minibatches**
- **구현 이유:** 준비된 transition rows에서 무작위 mini-batch를 뽑는다.
- **읽을 코드:** 아래 `evaluation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
portfolio_metrics = portfolio_agent.evaluate(q_star)
assert portfolio_metrics

with torch.no_grad():
    learned = online(encode_device(torch.arange(n_states)))
    s, a, r, ns, d, q_ref = ACCELERATOR.move(
        states, actions, rewards, next_states, dones, q_star
    )
    pred = learned[s, a]
    y = r + gamma * (1 - d) * learned[ns].max(1).values
    residual = float((pred - y).abs().mean())
    nonterminal = torch.arange(n_states, device=DEVICE) != terminal_state
    agreement = float(
        (learned.argmax(1)[nonterminal] == q_ref.argmax(1)[nonterminal]).float().mean()
    )
print(f"Bellman MAE={residual:.4f}, greedy agreement={agreement:.3f}")
assert math.isfinite(residual) and agreement >= 0.5
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].plot(history)
axes[0].set_title("replay TD loss")
axes[1].imshow(learned.detach().cpu().T, aspect="auto", cmap="viridis")
axes[1].set(title="Q(s,a)", xlabel="state", ylabel="action")
fig.tight_layout()
plt.show()

회귀 목표가 online Q와 동시에 빠르게 움직이면 bootstrap feedback이 불안정해집니다. 일정 기간 고정된 target은 supervised-like objective를 제공합니다.

### 단계 5. 통합 Agent가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `Methods, ‘Deep reinforcement learning’, Eq. (1)–(2)`의 **squared TD loss and frozen target**
- **핵심 식:**

  $$
  y_t=r_t+\gamma(1-d_t)\max_{a'}Q_{\theta^-}(s_{t+1},a'),\qquad \mathcal L=\mathbb E[(y_t-Q_\theta(s_t,a_t))^2]
  $$

- **입출력 shape:** `one-hot state [B, S] → Q [B, A]; action/reward/done [B]`
- **구현 이유:** 앞 셀은 target, objective, search를 작은 연산으로
  분해해 확인하고, 이 셀은 같은 구성 요소를
  `DqnAgent`에 주입해 최종 학습·평가 경로를
  하나로 묶습니다. r+γmax Q_target로 online Q를 회귀한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 정책·가치 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
config_contract = portfolio_agent.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(history) > 0
assert np.isfinite(np.asarray(history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_agent).__name__,
        "steps": len(history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 raw Atari frames, convolutional Q-network, 대규모 replay를 사용한다. 축소판은 동일한 target/replay mechanics를 작은 MDP에 적용한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.